In [1]:
import os
import mne
import numpy as np
import pandas as pd
from scipy import interpolate
import matplotlib.pyplot as plt
import scipy
import sklearn
from scipy.signal import resample
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [2]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [3]:
# root dir
root = 'PEARL-Neuro/'
# participants file path
participants_path = os.path.join(root, 'participants.tsv')
participants = pd.read_csv(participants_path, sep='\t')
participants

,participant_id,second_phase,session_order,APOE_rs429358,APOE_rs7412,APOE_haplotype,PICALM_rs3851179,age,sex,education,...,lymphocytes_%,monocytes_%,eosinophils_%,basophils_%,total_cholesterol,cholesterol_HDL,non-HDL_cholesterol,LDL_cholesterol,triglycerides,HSV_r
0,sub-01,1,1.0,T/T,C/C,e3/e3,A/A,59,0,3.0,...,26.6,10.4,0.8,0.3,174.3,37.9,136.4,100.48,179.6,1.0
1,sub-02,1,1.0,T/T,C/C,e3/e3,G/A,56,0,3.0,...,30.9,12.2,2.9,1.6,163.4,46.1,117.3,84.56,163.7,1.0
2,sub-03,1,0.0,T/T,C/C,e3/e3,G/A,52,0,3.0,...,36.1,9.0,2.1,0.6,152.9,43.3,109.6,100.88,43.6,1.0
3,sub-04,1,0.0,T/T,C/C,e3/e3,A/A,52,1,3.0,...,35.5,12.1,1.8,0.8,253.8,83.2,170.6,154.30,81.5,0.0
4,sub-05,1,0.0,T/T,C/C,e3/e3,G/A,52,1,3.0,...,32.9,8.1,1.2,0.9,283.1,66.1,217.0,188.28,143.6,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,sub-95,0,NaN,T/T,C/C,e3/e3,G/G,51,0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,sub-96,0,NaN,T/T,C/C,e3/e3,G/A,55,1,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189,sub-97,0,NaN,T/T,C/C,e3/e3,G/A,58,1,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190,sub-98,0,NaN,T/T,C/T,e3/e2,G/A,50,1,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Features

In [9]:
"""# Test for bad channels, sampling freq and shape
# only check the resting-state data
bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        for file in os.listdir(sub_path):
            if '.vhdr' in file and 'rest' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_brainvision(file_path, preload=True)
                # print(raw.get_montage())
                # get bad channels
                # print(raw.info['ch_names'])
                bad_channel = raw.info['bads']
                bad_channel_list.append(bad_channel)
                # get sampling frequency
                sampling_freq = raw.info['sfreq']
                sampling_freq_list.append(sampling_freq)
                # get eeg data
                data = raw.get_data()
                data_shape = data.shape
                data_shape_list.append(data_shape)"""

Extracting parameters from PEARL-Neuro/sub-01\eeg/sub-01_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 661519  =      0.000 ...   661.519 secs...
Extracting parameters from PEARL-Neuro/sub-02\eeg/sub-02_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 637719  =      0.000 ...   637.719 secs...
Extracting parameters from PEARL-Neuro/sub-03\eeg/sub-03_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 670759  =      0.000 ...   670.759 secs...
Extracting parameters from PEARL-Neuro/sub-04\eeg/sub-04_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 704099  =      0.000 ...   704.099 secs...
Extracting parameters from PEARL-Neuro/sub-05\eeg/sub-05_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 644859  =      0.000 ...   644.859 secs...
Extracting parameters from PEARL-Neuro/sub-06\eeg/sub-06_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 632819  =      0.000 ..

In [10]:
"""from collections import Counter

print(bad_channel_list)
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))"""

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
Channel number counter: Counter({127: 78})
Sampling rate counter: Counter({1000.0: 78})


## Features

In [4]:
def data_preprocessing(
    raw: mne.io.Raw,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 40.0,
    do_bad_interp: bool = True,
    verbose: bool = True,
):
    """
    Preprocessing steps ：
      2) Set Montage 
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（在 1 Hz 高通的副本上拟合，自动剔除眼动/肌电等分量，需 mne-icalabel）
      8) downsample to 250 Hz
    """
    
    # 1. Remove unknown channel
    unknown_channel = ['O9', 'O10']
    channel_list = list(raw.ch_names)
    new_channel_list = list(set(channel_list) - set(unknown_channel))
    raw.pick(new_channel_list)
        
    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1005'))
    if verbose:
        print("✔ Step 2, Montage set: 'standard_1005'.")
        
    # 3. Notch（工频）
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch @ {notch_freq} Hz")
        
    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")
        
    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bads: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)")
            
    # 6) Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: Average reference")
    
    # 5. ICA + ICLabel
    raw_ica = raw.copy()
    ica = ICA(n_components=None, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")
        
    return raw

In [5]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [6]:
# Loop through each subject and session to preprocess the EEG data
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
# For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
# So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")

sub_id = 1
for sub in os.listdir(root):
    if 'sub-' in sub:
        li_sub = []
        sub_path = os.path.join(root, sub, 'eeg/')
        print(sub_path)
        for file in os.listdir(sub_path):
            if '.vhdr' in file and 'rest' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_brainvision(file_path, preload=True)
                # For here, T7, T8 is close to T3, T4; P7, P8 is the same to T5, T6;
                # So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
                standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
                raw.pick(standard_channels)
                raw.reorder_channels(standard_channels)
                T_raw = raw.n_times
                original_fs = int(raw.info['sfreq'])
                for fs in SAMPLE_RATE_LIST:
                    T_res = int(T_raw * fs / original_fs)
                    # compute number of segments
                    n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                    subject_segment_counts[sub_id][fs] += n_seg
                    N_total += n_seg
                    print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                    print("----------------------------------------")
        sub_id += 1
    print("-------------------------------------\n")

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

PEARL-Neuro/sub-01\eeg/
Extracting parameters from PEARL-Neuro/sub-01\eeg/sub-01_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 661519  =      0.000 ...   661.519 secs...
fs=200Hz: T_res=132304, STEP=200, n_seg=660
----------------------------------------
fs=100Hz: T_res=66152, STEP=200, n_seg=329
----------------------------------------
fs=50Hz: T_res=33076, STEP=200, n_seg=164
----------------------------------------
-------------------------------------

PEARL-Neuro/sub-02\eeg/
Extracting parameters from PEARL-Neuro/sub-02\eeg/sub-02_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 637719  =      0.000 ...   637.719 secs...
fs=200Hz: T_res=127544, STEP=200, n_seg=636
--

In [7]:
output_root = os.path.join('Processed', sub_folder_path, 'PEARL-Neuro')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\PEARL-Neuro\X.dat
y path: Processed\L400\PEARL-Neuro\y.dat


## PASS 2: Process and store segments into memmap

In [8]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation

sub_id = 1
for sub in os.listdir(root):
    if 'sub-' in sub:
        li_sub = []
        sub_path = os.path.join(root, sub, 'eeg/')
        print(sub_path)
        label = 0 
        for file in os.listdir(sub_path):
            if '.vhdr' in file and 'rest' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_brainvision(file_path, preload=True)
                # For here, T7, T8 is close to T3, T4; P7, P8 is the same to T5, T6;
                # So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
                standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
                raw.pick(standard_channels)
                raw.reorder_channels(standard_channels)
                T_raw = raw.n_times
                original_fs = int(raw.info['sfreq'])
                data_raw = raw.get_data().T  # (T_raw, C)
                total_seconds_all += data_raw.shape[0] / original_fs
                for fs in SAMPLE_RATE_LIST:
                    # resample to target fs
                    data = resample_time_series(data_raw, original_fs, fs)
                    T_res, _ = data.shape
                    print(f"fs={fs}Hz: resampled shape={data.shape}")
        
                    # overlapped sliding window with fixed STEP (in timestamps)
                    starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                    print(f"fs={fs}Hz: segments={len(starts)}")
        
                    for s in starts:
                        if cur >= N_total:
                            raise RuntimeError("Exceeded predicted N_total.")
        
                        X_mm[cur] = data[s:s + SEG_LEN]   # (SEG_LEN, C)
                        y_mm[cur, 0] = float(label)       # label
                        y_mm[cur, 1] = float(sub_id)      # Global session ID
                        y_mm[cur, 2] = float(fs)          # sample rate (scale)
                        cur += 1
        sub_id += 1
    print("-------------------------------------\n")
    

total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

PEARL-Neuro/sub-01\eeg/
Extracting parameters from PEARL-Neuro/sub-01\eeg/sub-01_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 661519  =      0.000 ...   661.519 secs...
fs=200Hz: resampled shape=(132304, 19)
fs=200Hz: segments=660
fs=100Hz: resampled shape=(66152, 19)
fs=100Hz: segments=329
fs=50Hz: resampled shape=(33076, 19)
fs=50Hz: segments=164
-------------------------------------

PEARL-Neuro/sub-02\eeg/
Extracting parameters from PEARL-Neuro/sub-02\eeg/sub-02_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 637719  =      0.000 ...   637.719 secs...
fs=200Hz: resampled shape=(127544, 19)
fs=200Hz: segments=636
fs=100Hz: resampled shape=(63772, 19)
fs=100Hz: segments=317
fs=50Hz: resampled shape=(31886, 1

## Load and check the processed data

In [9]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 90147
  T = 400
  C = 19
  X_path = Processed\L400\PEARL-Neuro\X.dat
  y_path = Processed\L400\PEARL-Neuro\y.dat
-------------------------------------
Subject ID 001: X shape=(1153, 400, 19), y shape=(1153, 3)
Subject ID 002: X shape=(1111, 400, 19), y shape=(1111, 3)
Subject ID 003: X shape=(1169, 400, 19), y shape=(1169, 3)
Subject ID 004: X shape=(1229, 400, 19), y shape=(1229, 3)
Subject ID 005: X shape=(1124, 400, 19), y shape=(1124, 3)
Subject ID 006: X shape=(1103, 400, 19), y shape=(1103, 3)
Subject ID 007: X shape=(1281, 400, 19), y shape=(1281, 3)
Subject ID 008: X shape=(1131, 400, 19), y shape=(1131, 3)
Subject ID 009: X shape=(1209, 400, 19), y shape=(1209, 3)
Subject ID 010: X shape=(1117, 400, 19), y shape=(1117, 3)
Subject ID 011: X shape=(1128, 400, 19), y shape=(1128, 3)
Subject ID 012: X shape=(1145, 400, 19), y shape=(1145, 3)
Subject ID 013: X shape=(1110, 400, 19), y shape=(1110, 3)
Subject ID 014: X shape=(1176, 400, 19), y shape=(1176, 3)
Subject ID 